## Libraries

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Preprocessing Data

In [19]:
csv_2013 = "./csvs/2013_Jan.csv"
csv_2014 = "./csvs/jan_2014.csv"

#Create Pandas dataframe of 2013 csv file
jan_2013 = pd.read_csv(csv_2013)

#Preprocessing of Data to Remove Rows without Black and White Players Known
#Filters out 218 Games
partial_jan_2013 = jan_2013[jan_2013["White"] != "?"]
filtered_jan_2013 = partial_jan_2013[partial_jan_2013["Black"] != "?"]

#Need to covert "WhiteElo" and "BlackElo" from object type to int type
filtered_jan_2013["WhiteElo"] = filtered_jan_2013["WhiteElo"].astype(int)
filtered_jan_2013["BlackElo"] = filtered_jan_2013["BlackElo"].astype(int)


#Create Pandas dataframe of 2014 csv file
jan_2014 = pd.read_csv(csv_2014)

# #Preprocessing of Data to Remove Rows without Black and White Players Known
# #Filters out 111 Games (More than half of what jan_2013 had interestingly enough)
partial_jan_2014 = jan_2014[jan_2014["White"] != "?"]
filtered_jan_2014 = partial_jan_2014[partial_jan_2014["Black"] != "?"]

# #Need to covert "WhiteElo" and "BlackElo" from object type to int type
filtered_jan_2014["WhiteElo"] = filtered_jan_2014["WhiteElo"].astype(int)
filtered_jan_2014["BlackElo"] = filtered_jan_2014["BlackElo"].astype(int)



/tmp/ipykernel_524/3444839294.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_jan_2013["WhiteElo"] = filtered_jan_2013["WhiteElo"].astype(int)
/tmp/ipykernel_524/3444839294.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_jan_2013["BlackElo"] = filtered_jan_2013["BlackElo"].astype(int)
/tmp/ipykernel_524/3444839294.py:18: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  jan_2014 = pd.read_csv(csv_2014)
/tmp/ipykernel_524/34448

## General Data Statistics - 2013 of January File

In [33]:
#Rows / Columns in frame
rows_cols = filtered_jan_2013.shape
num_rows = rows_cols[0]
num_cols = rows_cols[1]

print("Number of Rows:", num_rows)
print("Number of Columns:", num_cols)

#Average of "WhiteElo", "BlackElo", and "Half Move Count" rounded up
avg_welo = np.ceil(np.mean(filtered_jan_2013["WhiteElo"].astype(int)))
avg_belo = np.ceil(np.mean(filtered_jan_2013["BlackElo"].astype(int)))
avg_turns = np.ceil(np.mean(filtered_jan_2013["Half Move Count"]))
print("Average White Player Elo:", avg_welo)
print("Average Black Player Elo:", avg_belo)
print("Average Number of Turns:",avg_turns)

#Unique Values of "Termination" Column
uterm_2013 = filtered_jan_2013["Termination"].unique()
print("Types of Game Terminations:", uterm_2013)

#Unique Values of "Result" Column
uresult_2013 = filtered_jan_2013["Result"].unique()
print("Types of Game Results:", uresult_2013)


Number of Rows: 121114
Number of Columns: 14
Average White Player Elo: 1606.0
Average Black Player Elo: 1596.0
Average Number of Turns: 68.0
Types of Game Terminations: ['Normal' 'Time forfeit']
Types of Game Results: ['1-0' '0-1' '1/2-1/2']


## General Data Statistics - 2014 of January File

In [86]:
#Rows / Columns in frame
rows_cols = filtered_jan_2014.shape
num_rows = rows_cols[0]
num_cols = rows_cols[1]

print("Number of Rows:", num_rows)
print("Number of Columns:", num_cols)

#Average of "WhiteElo", "BlackElo", and "Half Move Count" rounded up
avg_welo = np.ceil(np.mean(filtered_jan_2014["WhiteElo"].astype(int)))
avg_belo = np.ceil(np.mean(filtered_jan_2014["BlackElo"].astype(int)))
avg_turns = np.ceil(np.mean(filtered_jan_2014["Half Move Count"]))
print("Average White Player Elo:", avg_welo)
print("Average Black Player Elo:", avg_belo)
print("Average Number of Turns:",avg_turns)

#Unique Values of "Termination" Column
uterm_2014 = filtered_jan_2014["Termination"].unique()
print("Types of Game Terminations:", uterm_2014)

#Unique Values of "Result" Column
uresult_2014 = filtered_jan_2014["Result"].unique()
print("Types of Game Results:", uresult_2014)


Number of Rows: 697489
Number of Columns: 14
Average White Player Elo: 1619.0
Average Black Player Elo: 1609.0
Average Number of Turns: 68.0
Types of Game Terminations: ['Time forfeit' 'Normal' 'Rules infraction']
Types of Game Results: ['0-1' '1-0' '1/2-1/2']


## Cause of Loss Across Elo Brackets 

In [90]:
# Elo bracket definitions (name, lower_bound, upper_bound) (Elo Brackets borrowed from Ethan's code)
ELO_BRACKETS = [
    ("World Champion",       2800,  np.inf),
    ("Super Grandmaster",    2700,  2799),
    ("Grandmaster (GM)",     2500,  2699),
    ("International Master (IM)", 2400, 2499),
    ("FIDE Master (FM)",     2300,  2399),
    ("National Master",      2200,  2299),
    ("Expert",               2000,  2199),
    ("Class A",              1800,  1999),
    ("Class B",              1600,  1799),
    ("Class C",              1400,  1599),
    ("Class D",              1200,  1399),
    ("Beginner",             -np.inf, 1199),
]

#Counts of Loss Types by Elo Bracket
#Beginner
Beginner_TF = 0
Beginner_N = 0
Beginner_RI = 0
#Class D
CD_TF = 0
CD_N = 0
CD_RI = 0
#Class C
CC_TF = 0
CC_N = 0
CC_RI = 0
#Class B
CB_TF = 0
CB_N = 0
CB_RI = 0
#Class A
CA_TF = 0
CA_N = 0
CA_RI = 0
#Expert
Ex_TF = 0
Ex_N = 0
Ex_RI = 0
#National Master
NM_TF = 0
NM_N = 0
NM_RI = 0
#FIDE Master
FM_TF = 0
FM_N = 0
FM_RI = 0
#International Master
IM_TF = 0
IM_N = 0
IM_RI = 0
#Grandmaster
GM_TF = 0
GM_N = 0
GM_RI = 0
#Super Grandmaster
SG_TF = 0
SG_N = 0
SG_RI = 0
#World Champion
WC_TF = 0
WC_N = 0
WC_RI = 0
#Ties
ties = 0
#Broken Variable for Testing
broken_game = 0

#Elo Bracket Match Function
#Takes WhiteElo, BlackElo, Game Results, and Termination Type
def elo_match(welo, belo, res, term):
    global Beginner_TF
    global Beginner_N
    global Beginner_RI
    global CD_TF
    global CD_N
    global CD_RI
    global CC_TF
    global CC_N
    global CC_RI
    global CB_TF
    global CB_N
    global CB_RI
    global CA_TF
    global CA_N
    global CA_RI
    global Ex_TF
    global Ex_N
    global Ex_RI
    global NM_TF
    global NM_N
    global NM_RI
    global FM_TF
    global FM_N
    global FM_RI
    global IM_TF
    global IM_N
    global IM_RI
    global GM_TF
    global GM_N
    global GM_RI
    global SG_TF
    global SG_N
    global SG_RI
    global WC_TF
    global WC_N
    global WC_RI
    global ties
    global broken_game
    match(welo, belo, res, term):
        #Beginner Cases
        case(_, belo, "1-0", "Time forfeit") if belo < 1200:
            Beginner_TF = Beginner_TF + 1
        case(_, belo, "1-0", "Normal") if belo < 1200:
            Beginner_N = Beginner_N + 1
        case(_, belo, "1-0", "Rules infraction") if belo < 1200:
            Beginner_RI = Beginner_RI + 1
        case(welo, _, "0-1", "Time forfeit") if welo < 1200:
            Beginner_TF = Beginner_TF + 1
        case(welo, _, "0-1", "Normal") if welo < 1200:
            Beginner_N = Beginner_N + 1
        case(welo, _, "0-1", "Rules infraction") if welo < 1200:
            Beginner_RI = Beginner_RI + 1
        #Class D Cases
        case(_, belo, "1-0", "Time forfeit") if 1200 <= belo < 1400:
            CD_TF = CD_TF + 1
        case(_, belo, "1-0", "Normal") if 1200 <= belo < 1400:
            CD_N = CD_N + 1
        case(_, belo, "1-0", "Rules infraction") if 1200 <= belo < 1400:
            CD_RI = CD_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 1200 <= welo < 1400:
            CD_TF = CD_TF + 1
        case(welo, _, "0-1", "Normal") if 1200 <= welo < 1400:
            CD_N = CD_N + 1
        case(welo, _, "0-1", "Rules infraction") if 1200 <= welo < 1400:
            CD_RI = CD_RI + 1
        #Class C Cases
        case(_, belo, "1-0", "Time forfeit") if 1400 <= belo < 1600:
            CC_TF = CC_TF + 1
        case(_, belo, "1-0", "Normal") if 1400 <= belo < 1600:
            CC_N = CC_N + 1
        case(_, belo, "1-0", "Rules infraction") if 1400 <= belo < 1600:
            CC_RI = CC_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 1400 <= welo < 1600:
            CC_TF = CC_TF + 1
        case(welo, _, "0-1", "Normal") if 1400 <= welo < 1600:
            CC_N = CC_N + 1
        case(welo, _, "0-1", "Rules infraction") if 1400 <= welo < 1600:
            CC_RI = CC_RI + 1
        #Class B Cases
        case(_, belo, "1-0", "Time forfeit") if 1600 <= belo < 1800:
            CB_TF = CB_TF + 1
        case(_, belo, "1-0", "Normal") if 1600 <= belo < 1800:
            CB_N = CB_N + 1
        case(_, belo, "1-0", "Rules infraction") if 1600 <= belo < 1800:
            CB_RI = CB_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 1600 <= welo < 1800:
            CB_TF = CB_TF + 1
        case(welo, _, "0-1", "Normal") if 1600 <= welo < 1800:
            CB_N = CB_N + 1
        case(welo, _, "0-1", "Rules infraction") if 1600 <= welo < 1800:
            CB_RI = CB_RI + 1
        #Class A Cases
        case(_, belo, "1-0", "Time forfeit") if 1800 <= belo < 2000:
            CA_TF = CA_TF + 1
        case(_, belo, "1-0", "Normal") if 1800 <= belo < 2000:
            CA_N = CA_N + 1
        case(_, belo, "1-0", "Rules infraction") if 1800 <= belo < 2000:
            CA_RI = CA_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 1800 <= welo < 2000:
            CA_TF = CA_TF + 1
        case(welo, _, "0-1", "Normal") if 1800 <= welo < 2000:
            CA_N = CA_N + 1
        case(welo, _, "0-1", "Rules infraction") if 1800 <= welo < 2000:
            CA_RI = CA_RI + 1
        #Expert Cases
        case(_, belo, "1-0", "Time forfeit") if 2000 <= belo < 2200:
            Ex_TF = Ex_TF + 1
        case(_, belo, "1-0", "Normal") if 2000 <= belo < 2200:
            Ex_N = Ex_N + 1
        case(_, belo, "1-0", "Rules infraction") if 2000 <= belo < 2200:
            Ex_RI = Ex_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 2000 <= welo < 2200:
            Ex_TF = Ex_TF + 1
        case(welo, _, "0-1", "Normal") if 2000 <= welo < 2200:
            Ex_N = Ex_N + 1
        case(welo, _, "0-1", "Rules infraction") if 2000 <= welo < 2200:
            Ex_RI = Ex_RI + 1
        #National Master Cases
        case(_, belo, "1-0", "Time forfeit") if 2200 <= belo < 2300:
            NM_TF = NM_TF + 1
        case(_, belo, "1-0", "Normal") if 2200 <= belo < 2300:
            NM_N = NM_N + 1
        case(_, belo, "1-0", "Rules infraction") if 2200 <= belo < 2300:
            NM_RI = NM_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 2200 <= welo < 2300:
            NM_TF = NM_TF + 1
        case(welo, _, "0-1", "Normal") if 2200 <= welo < 2300:
            NM_N = NM_N + 1
        case(welo, _, "0-1", "Rules infraction") if 2200 <= welo < 2300:
            NM_RI = NM_RI + 1
        #FIDE Master Cases
        case(_, belo, "1-0", "Time forfeit") if 2300 <= belo < 2400:
            FM_TF = FM_TF + 1
        case(_, belo, "1-0", "Normal") if 2300 <= belo < 2400:
            FM_N = FM_N + 1
        case(_, belo, "1-0", "Rules infraction") if 2300 <= belo < 2400:
            FM_RI = FM_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 2300 <= welo < 2400:
            FM_TF = FM_TF + 1
        case(welo, _, "0-1", "Normal") if 2300 <= welo < 2400:
            FM_N = FM_N + 1
        case(welo, _, "0-1", "Rules infraction") if 2300 <= welo < 2400:
            FM_RI = FM_RI + 1
        #International Master Cases
        case(_, belo, "1-0", "Time forfeit") if 2400 <= belo < 2500:
            IM_TF = IM_TF + 1
        case(_, belo, "1-0", "Normal") if 2400 <= belo < 2500:
            IM_N = IM_N + 1
        case(_, belo, "1-0", "Rules infraction") if 2400 <= belo < 2500:
            IM_RI = IM_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 2400 <= welo < 2500:
            IM_TF = IM_TF + 1
        case(welo, _, "0-1", "Normal") if 2400 <= welo < 2500:
            IM_N = IM_N + 1
        case(welo, _, "0-1", "Rules infraction") if 2400 <= welo < 2500:
            IM_RI = IM_RI + 1
        #Grandmaster Cases
        case(_, belo, "1-0", "Time forfeit") if 2500 <= belo < 2700:
            GM_TF = GM_TF + 1
        case(_, belo, "1-0", "Normal") if 2500 <= belo < 2700:
            GM_N = GM_N + 1
        case(_, belo, "1-0", "Rules infraction") if 2500 <= belo < 2700:
            GM_RI = GM_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 2500 <= welo < 2700:
            GM_TF = GM_TF + 1
        case(welo, _, "0-1", "Normal") if 2500 <= welo < 2700:
            GM_N = GM_N + 1
        case(welo, _, "0-1", "Rules infraction") if 2500 <= welo < 2700:
            GM_RI = GM_RI + 1
        #Super Grandmaster Cases
        case(_, belo, "1-0", "Time forfeit") if 2700 <= belo < 2800:
            SG_TF = SG_TF + 1
        case(_, belo, "1-0", "Normal") if 2700 <= belo < 2800:
            SG_N = SG_N + 1
        case(_, belo, "1-0", "Rules infraction") if 2700 <= belo < 2800:
            SG_RI = SG_RI + 1
        case(welo, _, "0-1", "Time forfeit") if 2700 <= welo < 2800:
            SG_TF = SG_TF + 1
        case(welo, _, "0-1", "Normal") if 2700 <= welo < 2800:
            SG_N = SG_N + 1
        case(welo, _, "0-1", "Rules infraction") if 2700 <= welo < 2800:
            SG_RI = SG_RI + 1
        #World Champion Cases
        case(_, belo, "1-0", "Time forfeit") if belo >= 2800:
            WC_TF = WC_TF + 1
        case(_, belo, "1-0", "Normal") if belo >= 2800:
            WC_N = WC_N + 1
        case(_, belo, "1-0", "Rules infraction") if belo >= 2800:
            WC_RI = WC_RI + 1
        case(welo, _, "0-1", "Time forfeit") if welo >= 2800:
            WC_TF = WC_TF + 1
        case(welo, _, "0-1", "Normal") if welo >= 2800:
            WC_N = WC_N + 1
        case(welo, _, "0-1", "Rules infraction") if welo >= 2800:
            WC_RI = WC_RI + 1
        #Ties and Broken Games Cases
        case(_, _, "1/2-1/2", _):
            ties = ties + 1
        case _:
            broken_game = broken_game + 1

#Counting Loss Per Category for Elo Ranks from Jan_2013 and Jan_2014 ("WhiteElo" is Col 7 and "BlackElo" is Col 8)
#Example of what each game is:
#Pandas(GameID=0, Event='Rated Classical game', White='BFG9k', Black='mamalak', Result='1-0', Date='2012.12.31', Time='23:01:03', WhiteElo=1639, BlackElo=1403, ECO='C00', Opening='French Defense: Normal Variation', TimeControl='600+8', Termination='Normal', _13=25)
for game in filtered_jan_2013.itertuples(index=False):
    elo_match(game[7], game[8], game[4], game[12])
    #print((game[7]), (game[8]), game[4], game[12])
    #elo_match(game[7], game[8], game[4], game[12])
    #elo_match(1660, 1421, game[4], game[12])
for game2 in filtered_jan_2014.itertuples(index=False):
    elo_match(game2[7], game2[8], game2[4], game2[12])
print(broken_game, CB_TF)

0 80692
